# 🐍 Python Expert Mastery — Deep Internals & Advanced Patterns

This notebook extends the basics notebook into true expert territory:

| # | Section |
|---|---|
| 7 | Memory Management & Garbage Collection |
| 8 | CPython Internals & Bytecode |
| 9 | Advanced Generators, Coroutines & Pipelines |
| 10 | Advanced asyncio |
| 11 | Performance & Profiling |
| 12 | Advanced Graph Algorithms |
| 13 | Advanced Tree Data Structures |
| 14 | Advanced Sorting, Searching & Bit Manipulation |
| 15 | Testing with pytest |
| 16 | Regular Expressions (Expert Level) |
| 17 | File I/O, Pathlib & Serialization |
| 18 | Logging (Production Grade) |
| 19 | Networking & Sockets |
| 20 | Advanced Type System |
| 21 | Structural Pattern Matching (Python 3.10+) |
| 22 | Advanced OOP — Mixins, Class Factories, Protocols |
| 23 | More Design Patterns |
| 24 | Advanced Dynamic Programming |
| 25 | Enum, NamedTuple & Advanced Built-ins |
| 26 | Weak References & The Import System |
| 27 | Python Best Practices, Idioms & Anti-Patterns |

---

# 🔬 SECTION 7 — Memory Management & Garbage Collection

Understanding how Python manages memory is essential for writing high-performance, leak-free code.

## 7.1 Reference Counting

In [ ]:
import sys
import gc
import ctypes

# CPython uses reference counting as its PRIMARY memory management strategy.
# Every object has a reference count. When it drops to 0, memory is freed immediately.

a = [1, 2, 3]          # refcount = 1
b = a                   # refcount = 2
c = [a, a]              # refcount = 4 (list stores 2 refs)

print(f"refcount of 'a' list: {sys.getrefcount(a)}")  # +1 because getrefcount arg is a ref

del b                   # refcount drops by 1
print(f"after del b: {sys.getrefcount(a)}")

del c                   # refcount drops by 2
print(f"after del c: {sys.getrefcount(a)}")

# ---- Inspecting live objects ----
x = object()
x_id = id(x)
print(f"id of x: {x_id}")
# After del x, the memory at x_id may be reused!

# ---- Memory size of objects ----
print(f"int(0) size:      {sys.getsizeof(0)} bytes")
print(f"int(2^30) size:   {sys.getsizeof(2**30)} bytes")
print(f"empty list size:  {sys.getsizeof([])} bytes")
print(f"list[1..10] size: {sys.getsizeof(list(range(10)))} bytes")
print(f"empty dict size:  {sys.getsizeof({})} bytes")
print(f"empty str size:   {sys.getsizeof('')} bytes")
print(f"'hello' size:     {sys.getsizeof('hello')} bytes")

In [ ]:
# ---- Cyclic Reference Problem ----
# Reference counting CANNOT handle cycles. The cyclic GC handles these.

class Node:
    def __init__(self, name):
        self.name = name
        self.ref = None

    def __del__(self):
        print(f"Node '{self.name}' deleted")

# Create a cycle: a → b → a
a = Node('A')
b = Node('B')
a.ref = b
b.ref = a

print(f"gc tracking: {gc.is_tracked(a)}")

# Delete our references — but the cycle keeps objects alive!
del a, b
print("Deleted a and b — waiting for GC...")

# Force the cyclic GC to run
collected = gc.collect()
print(f"GC collected {collected} objects")

# ---- GC Generations ----
# Python's GC uses 3 generations (young, middle-aged, old).
# New objects start in gen 0. Survivors move to gen 1, then gen 2.
print(f"\nGC thresholds: {gc.get_threshold()}")   # (700, 10, 10)
print(f"GC counts:     {gc.get_count()}")          # Objects in each generation

# ---- Disabling GC (careful!) ----
# gc.disable()  # Use only in performance-critical sections
# ... code with many short-lived objects ...
# gc.enable()
# gc.collect()  # Manual collection

## 7.2 Weak References

In [ ]:
import weakref

# A weak reference does NOT increment the reference count.
# The object can be garbage collected even if weak refs exist.
# Use case: caches, callbacks, circular data structures without memory leaks.

class ExpensiveObject:
    def __init__(self, data):
        self.data = data

    def __del__(self):
        print(f"ExpensiveObject({self.data!r}) destroyed")

obj = ExpensiveObject('important data')

# Create a weak reference
weak = weakref.ref(obj)

print(f"weak ref alive: {weak()}")    # Dereference: returns obj or None
print(f"is alive: {weak() is not None}")

del obj    # Only strong reference deleted
print(f"after del: {weak()}")   # None — object was GC'd

# ---- weakref.WeakValueDictionary — cache that doesn't prevent GC ----
import weakref

class Cache:
    """LRU-like cache using weak values."""
    def __init__(self):
        self._cache = weakref.WeakValueDictionary()

    def get_or_create(self, key, factory):
        if key in self._cache:
            return self._cache[key]
        obj = factory(key)
        self._cache[key] = obj
        return obj

cache = Cache()
obj1 = cache.get_or_create('item1', lambda k: ExpensiveObject(k))
print(f"cache size: {len(cache._cache)}")  # 1
del obj1
gc.collect()
print(f"cache size after del: {len(cache._cache)}")  # 0

# ---- Callbacks on weakref death ----
def on_finalize(ref):
    print(f"Object referenced by {ref} was garbage collected!")

obj2 = ExpensiveObject('temp')
weak2 = weakref.ref(obj2, on_finalize)  # Callback when obj dies
del obj2

## 7.3 `__del__`, Finalizers & `weakref.finalize`

In [ ]:
# __del__ is called when an object's refcount hits 0.
# WARNING: __del__ timing is non-deterministic with cycles.
# Prefer context managers for resource cleanup!

# ---- weakref.finalize — safer than __del__ ----
class Connection:
    def __init__(self, host):
        self.host = host
        self._finalizer = weakref.finalize(
            self, Connection._cleanup, host  # Note: no 'self' — avoids circular ref
        )
        print(f"Connected to {host}")

    @staticmethod
    def _cleanup(host):
        print(f"Connection to {host} cleaned up")

    def close(self):
        self._finalizer()   # Explicit cleanup

conn = Connection('localhost:5432')
conn.close()   # Explicit cleanup
del conn       # _cleanup NOT called again (finalize runs only once)

# ---- Memory profiling trick ----
import tracemalloc

tracemalloc.start()

# Snapshot before
snapshot1 = tracemalloc.take_snapshot()

# Allocate memory
big_list = [dict(x=i, y=i*2) for i in range(10_000)]

# Snapshot after
snapshot2 = tracemalloc.take_snapshot()

top_stats = snapshot2.compare_to(snapshot1, 'lineno')
for stat in top_stats[:3]:
    print(stat)

del big_list
tracemalloc.stop()

---
# 🔩 SECTION 8 — CPython Internals & Bytecode

Understanding how CPython compiles and executes Python code.

## 8.1 The `dis` Module — Bytecode Disassembly

In [ ]:
import dis

# Python compiles source code to BYTECODE before execution.
# dis lets you inspect the bytecode instructions.

def simple_add(a, b):
    return a + b

dis.dis(simple_add)
print()

# Compare list append vs list comprehension
def build_list_append():
    result = []
    for i in range(10):
        result.append(i * 2)
    return result

def build_list_comp():
    return [i * 2 for i in range(10)]

print("=== append version ===")
dis.dis(build_list_append)
print("\n=== comprehension version ===")
dis.dis(build_list_comp)
# Comprehension generates LIST_APPEND which is implemented as C-level loop — faster!

In [ ]:
# ---- Code Objects ----
# Every function, class, module compiles to a 'code object'.

def example(x, y=10, *args, **kwargs):
    z = x + y
    return z

code = example.__code__

print(f"Function name:     {code.co_name}")
print(f"Argument count:    {code.co_argcount}")
print(f"Local variables:   {code.co_varnames}")
print(f"Constants:         {code.co_consts}")
print(f"Free variables:    {code.co_freevars}")   # Captured from enclosing scope
print(f"Stack size needed: {code.co_stacksize}")
print(f"File:              {code.co_filename}")
print(f"Line number:       {code.co_firstlineno}")
print(f"Bytecode bytes:    {code.co_code!r}")

# ---- Compile & exec at runtime ----
source = """
def greet(name):
    return f'Hello, {name}!'
result = greet('World')
"""
namespace = {}
compiled = compile(source, '<string>', 'exec')
exec(compiled, namespace)
print(namespace['result'])  # Hello, World!

# eval for expressions
expr_result = eval(compile('2 ** 10 + len([1,2,3])', '<expr>', 'eval'))
print(expr_result)  # 1027

## 8.2 The `__dunder__` Data Model Internals

In [ ]:
# How Python REALLY works: every operator calls a dunder method.
# Understanding this lets you fully control Python's behaviour.

class SmartInt:
    """Custom integer with full numeric protocol."""

    def __init__(self, val): self.val = val

    # Comparison — return NotImplemented to let Python try reflected op
    def __eq__(self, other):
        if isinstance(other, SmartInt): return self.val == other.val
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, SmartInt): return self.val < other.val
        return NotImplemented

    # Python derives __gt__, __le__, __ge__, __ne__ from __eq__ + __lt__
    # if you use @functools.total_ordering

    # Numeric protocol
    def __add__(self, other):
        if isinstance(other, SmartInt): return SmartInt(self.val + other.val)
        if isinstance(other, int):      return SmartInt(self.val + other)
        return NotImplemented

    def __radd__(self, other): return self.__add__(other)  # other + self
    def __iadd__(self, other):  # self += other
        result = self.__add__(other)
        if result is NotImplemented: return result
        self.val = result.val
        return self

    # Hash — required when __eq__ is defined
    def __hash__(self): return hash(self.val)

    # Context manager protocol
    def __enter__(self): return self
    def __exit__(self, *args): return False

    # Boolean protocol
    def __bool__(self): return self.val != 0

    # Numeric conversions
    def __int__(self):   return self.val
    def __float__(self): return float(self.val)
    def __index__(self): return self.val  # For slicing, bin(), hex(), oct()

    def __repr__(self): return f"SmartInt({self.val})"

a = SmartInt(10)
b = SmartInt(5)
print(a + b)          # SmartInt(15)
print(100 + a)        # SmartInt(110) — __radd__
a += SmartInt(3)
print(a)              # SmartInt(13)
print(bin(b))         # 0b101 — uses __index__
print(bool(SmartInt(0)))  # False

In [ ]:
# ---- __getattr__ vs __getattribute__ ----
# __getattribute__: called on EVERY attribute access (dangerous to override)
# __getattr__: called ONLY when normal lookup fails (safe fallback)

class DynamicAttributes:
    """
    Stores unknown attributes in a backing dict.
    Models something like a flexible config object.
    """
    def __init__(self):
        # Use object.__setattr__ to avoid triggering our __setattr__
        object.__setattr__(self, '_data', {})

    def __getattr__(self, name):
        # Only called when normal lookup fails
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(f"'{type(self).__name__}' has no attribute '{name}'")

    def __setattr__(self, name, value):
        if name.startswith('_'):
            object.__setattr__(self, name, value)  # Private: normal behavior
        else:
            self._data[name] = value

    def __delattr__(self, name):
        if name in self._data:
            del self._data[name]
        else:
            object.__delattr__(self, name)

    def __contains__(self, name):
        return name in self._data

cfg = DynamicAttributes()
cfg.host    = 'localhost'
cfg.port    = 5432
cfg.timeout = 30

print(cfg.host, cfg.port)  # localhost 5432
print('host' in cfg)       # True
del cfg.host
print(cfg._data)            # {'port': 5432, 'timeout': 30}

---
# 🔄 SECTION 9 — Advanced Generators, Coroutines & Pipelines

## 9.1 Generator Pipelines

In [ ]:
# Generators compose beautifully as lazy data pipelines.
# Each stage processes one item at a time — O(1) memory regardless of data size!

import os
import re

# ---- Pipeline stages ----
def read_lines(text: str):
    """Stage 1: Source — emit lines one at a time."""
    for line in text.splitlines():
        yield line

def strip_comments(lines, comment='#'):
    """Stage 2: Filter — remove comment lines."""
    for line in lines:
        line = line.strip()
        if line and not line.startswith(comment):
            yield line

def parse_kv(lines, sep='='):
    """Stage 3: Transform — parse key=value pairs."""
    for line in lines:
        if sep in line:
            key, _, value = line.partition(sep)
            yield key.strip(), value.strip()

def type_coerce(pairs):
    """Stage 4: Transform — coerce values to Python types."""
    for key, value in pairs:
        if value.isdigit():          coerced = int(value)
        elif value.replace('.','',1).isdigit(): coerced = float(value)
        elif value.lower() == 'true': coerced = True
        elif value.lower() == 'false': coerced = False
        else: coerced = value
        yield key, coerced

config_text = """
# Database config
host = localhost
port = 5432
debug = true
timeout = 30.5
name = my_app
"""

# Build the pipeline — nothing executed yet!
pipeline = type_coerce(parse_kv(strip_comments(read_lines(config_text))))

# Materialize — processes one item at a time
config = dict(pipeline)
print(config)  # {'host': 'localhost', 'port': 5432, 'debug': True, ...}

In [ ]:
# ---- Generator-based Coroutines (pre-async/await) ----
# These are still useful for understanding how asyncio works internally.

def running_stats():
    """
    Coroutine that computes running mean and variance using Welford's algorithm.
    send() values in; yield (mean, variance) out.
    """
    count = 0
    mean  = 0.0
    M2    = 0.0

    while True:
        value = yield (mean, M2 / count if count > 1 else 0.0)
        if value is None:
            return
        count  += 1
        delta   = value - mean
        mean   += delta / count
        M2     += delta * (value - mean)

stats = running_stats()
next(stats)  # Prime

data = [2, 4, 4, 4, 5, 5, 7, 9]
for x in data:
    mean, var = stats.send(x)
    print(f"After {x:2d}: mean={mean:.2f}, variance={var:.2f}")

# ---- throw() — inject exceptions into generators ----
def resilient_gen():
    for i in range(10):
        try:
            yield i
        except ValueError as e:
            print(f"Caught in generator: {e}, continuing")
            yield -1  # Recovery value

gen = resilient_gen()
print(next(gen))               # 0
print(gen.throw(ValueError, 'bad value'))  # Caught in generator, yields -1
print(next(gen))               # 1

# ---- close() — terminate a generator ----
def infinite():
    try:
        i = 0
        while True:
            yield i
            i += 1
    except GeneratorExit:
        print("Generator closed gracefully")

g = infinite()
print([next(g) for _ in range(5)])  # [0,1,2,3,4]
g.close()  # Sends GeneratorExit into the generator

In [ ]:
# ---- Generator return values ----
# Generators can RETURN a value (accessible via StopIteration.value)

def countdown(n):
    while n > 0:
        yield n
        n -= 1
    return 'blastoff!'  # Return value

gen = countdown(3)
while True:
    try:
        print(next(gen))
    except StopIteration as e:
        print(f"Generator returned: {e.value}")
        break

# ---- yield from delegates return value ----
def outer():
    result = yield from countdown(3)  # Captures inner return value
    print(f"Inner returned: {result}")
    yield 'done'

print(list(outer()))

---
# ⚡ SECTION 10 — Advanced asyncio

In [ ]:
import asyncio
import time
from typing import Optional

# ---- Timeout & Cancellation ----
async def slow_api_call(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"Result from {name}"

async def with_timeout_demo():
    # asyncio.timeout (Python 3.11+)
    try:
        async with asyncio.timeout(0.3):
            result = await slow_api_call('service', 1.0)  # Too slow!
    except asyncio.TimeoutError:
        print("Request timed out!")

    # asyncio.wait_for — older style
    try:
        result = await asyncio.wait_for(slow_api_call('fast', 0.1), timeout=0.5)
        print(f"Got: {result}")
    except asyncio.TimeoutError:
        print("Timed out!")

await with_timeout_demo()

In [ ]:
# ---- asyncio.Semaphore — limit concurrent coroutines ----
async def rate_limited_demo():
    sem = asyncio.Semaphore(3)  # Max 3 concurrent requests
    results = []

    async def fetch(n):
        async with sem:
            await asyncio.sleep(0.1)  # Simulate I/O
            return n * n

    tasks = [asyncio.create_task(fetch(i)) for i in range(10)]
    results = await asyncio.gather(*tasks)
    print(f"Results: {results}")

await rate_limited_demo()

# ---- asyncio.Event, Lock, Condition ----
async def event_demo():
    event = asyncio.Event()

    async def waiter(name):
        print(f"{name} waiting...")
        await event.wait()
        print(f"{name} proceeding!")

    async def trigger():
        await asyncio.sleep(0.1)
        print("Triggering event")
        event.set()

    await asyncio.gather(
        waiter('W1'), waiter('W2'), waiter('W3'), trigger()
    )

await event_demo()

In [ ]:
# ---- Task management: cancel, shield, callbacks ----
async def task_management_demo():
    # Create task explicitly
    task = asyncio.create_task(slow_api_call('main', 1.0), name='main-task')

    await asyncio.sleep(0.1)
    task.cancel()  # Cancel it

    try:
        await task
    except asyncio.CancelledError:
        print(f"Task '{task.get_name()}' was cancelled")

    # asyncio.shield — protect a coroutine from cancellation
    async def critical_cleanup():
        await asyncio.sleep(0.05)
        print("Critical cleanup done")

    protected = asyncio.shield(critical_cleanup())
    await protected

await task_management_demo()

# ---- asyncio.wait — finer control than gather ----
async def wait_demo():
    tasks = [
        asyncio.create_task(slow_api_call('A', 0.3)),
        asyncio.create_task(slow_api_call('B', 0.1)),
        asyncio.create_task(slow_api_call('C', 0.5)),
    ]

    # FIRST_COMPLETED: proceed when any task finishes
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)

    for t in done:
        print(f"First done: {t.result()}")

    # Cancel remaining
    for t in pending:
        t.cancel()

await wait_demo()

In [ ]:
# ---- Full async producer-consumer with backpressure ----
async def async_producer_consumer():
    queue = asyncio.Queue(maxsize=5)  # Backpressure: producers block when full
    SENTINEL = object()

    async def producer(name: str, n_items: int):
        for i in range(n_items):
            await asyncio.sleep(0.02)
            item = f"{name}:{i}"
            await queue.put(item)  # Blocks if queue is full — backpressure!
            print(f"Produced: {item} (queue size: {queue.qsize()})")
        await queue.put(SENTINEL)

    async def consumer(name: str):
        while True:
            item = await queue.get()
            if item is SENTINEL:
                queue.task_done()
                break
            await asyncio.sleep(0.05)  # Consumer is slower than producer
            print(f"{name} consumed: {item}")
            queue.task_done()

    await asyncio.gather(
        producer('P', 6),
        consumer('C1'),
    )
    await queue.join()  # Wait until all items processed
    print("All items processed")

await async_producer_consumer()

---
# 📊 SECTION 11 — Performance & Profiling

Write code that's not just correct but fast.

In [ ]:
import cProfile
import pstats
import io
import timeit
from functools import wraps

# ---- cProfile — function-level profiler ----
def fibonacci_slow(n):
    if n <= 1: return n
    return fibonacci_slow(n-1) + fibonacci_slow(n-2)

# Profile to a string buffer
pr = cProfile.Profile()
pr.enable()
fibonacci_slow(25)
pr.disable()

stream = io.StringIO()
ps = pstats.Stats(pr, stream=stream).sort_stats('cumulative')
ps.print_stats(5)   # Top 5 functions
print(stream.getvalue())

# ---- timeit — micro-benchmarks ----
# Comparing list construction methods
methods = {
    'list()':       "list(range(1000))",
    '[*range()]':   "[*range(1000)]",
    'comprehension': "[i for i in range(1000)]",
    'append loop':   "r=[]; [r.append(i) for i in range(1000)]",
}

for name, stmt in methods.items():
    t = timeit.timeit(stmt, number=10_000)
    print(f"{name:20s}: {t*1000:.2f} ms for 10k runs")

In [ ]:
# ---- Optimization techniques ----
import timeit

# 1. Local variable lookup is faster than global
def sum_global(n):
    total = 0
    for i in range(n):
        total += i
    return total

def sum_local(n):
    total = 0
    _range = range   # Cache built-in as local
    for i in _range(n):
        total += i
    return total

# 2. Set membership is O(1), list is O(n)
data_list = list(range(10_000))
data_set  = set(data_list)

list_time = timeit.timeit(lambda: 9999 in data_list, number=100_000)
set_time  = timeit.timeit(lambda: 9999 in data_set,  number=100_000)
print(f"List lookup: {list_time:.4f}s")
print(f"Set  lookup: {set_time:.4f}s")
print(f"Speedup: {list_time/set_time:.0f}x")

# 3. String joining — join is much faster than += in loops
words = ['word'] * 1000

def concat_plus():
    s = ''
    for w in words: s += w
    return s

def concat_join():
    return ''.join(words)

t1 = timeit.timeit(concat_plus, number=1000)
t2 = timeit.timeit(concat_join, number=1000)
print(f"str += : {t1:.4f}s")
print(f"join:    {t2:.4f}s")
print(f"Speedup: {t1/t2:.0f}x")

In [ ]:
import sys

# ---- __slots__ memory/speed improvement ----
class WithDict:
    def __init__(self, x, y, z): self.x, self.y, self.z = x, y, z

class WithSlots:
    __slots__ = ('x', 'y', 'z')
    def __init__(self, x, y, z): self.x, self.y, self.z = x, y, z

n = 100_000
dict_objs  = [WithDict(i, i+1, i+2) for i in range(n)]
slots_objs = [WithSlots(i, i+1, i+2) for i in range(n)]

dict_mem  = sum(sys.getsizeof(o) + sys.getsizeof(o.__dict__) for o in dict_objs[:100]) * 1000
slots_mem = sum(sys.getsizeof(o) for o in slots_objs[:100]) * 1000

print(f"Dict objects  (100k): ~{dict_mem/1024:.0f} KB")
print(f"Slots objects (100k): ~{slots_mem/1024:.0f} KB")
print(f"Memory saving: {1 - slots_mem/dict_mem:.0%}")

# ---- array module — typed arrays, much faster than lists for numerics ----
import array
import timeit

# Regular list of floats
py_list  = list(range(100_000))
# Typed array of signed ints — more compact, faster for numeric ops
int_arr  = array.array('l', range(100_000))

t1 = timeit.timeit(lambda: sum(py_list), number=1000)
t2 = timeit.timeit(lambda: sum(int_arr), number=1000)
print(f"\nlist sum: {t1:.4f}s")
print(f"array sum: {t2:.4f}s")

---
# 🌐 SECTION 12 — Advanced Graph Algorithms

Graphs power social networks, routing, dependency resolution, and much more.

In [ ]:
from collections import defaultdict, deque
from typing import Optional
import heapq

class Graph:
    """Adjacency-list graph supporting weighted/unweighted directed/undirected."""

    def __init__(self, directed=True):
        self.adj = defaultdict(list)  # node -> [(neighbor, weight)]
        self.directed = directed

    def add_edge(self, u, v, weight=1):
        self.adj[u].append((v, weight))
        if not self.directed:
            self.adj[v].append((u, weight))

    def nodes(self):
        return set(self.adj.keys()) | {v for edges in self.adj.values() for v, _ in edges}

    # ---- BFS — Shortest path in unweighted graph ----
    def bfs(self, start) -> dict:
        """Returns {node: distance} from start."""
        dist = {start: 0}
        queue = deque([start])
        while queue:
            node = queue.popleft()
            for neighbor, _ in self.adj[node]:
                if neighbor not in dist:
                    dist[neighbor] = dist[node] + 1
                    queue.append(neighbor)
        return dist

    def shortest_path_bfs(self, start, end) -> Optional[list]:
        """Returns shortest path as list of nodes."""
        parent = {start: None}
        queue  = deque([start])
        while queue:
            node = queue.popleft()
            if node == end:
                path = []
                while node is not None:
                    path.append(node)
                    node = parent[node]
                return path[::-1]
            for neighbor, _ in self.adj[node]:
                if neighbor not in parent:
                    parent[neighbor] = node
                    queue.append(neighbor)
        return None  # No path

    # ---- DFS — Iterative & Recursive ----
    def dfs_iterative(self, start) -> list:
        visited, stack, order = set(), [start], []
        while stack:
            node = stack.pop()
            if node not in visited:
                visited.add(node)
                order.append(node)
                for neighbor, _ in reversed(self.adj[node]):
                    if neighbor not in visited:
                        stack.append(neighbor)
        return order

    def dfs_recursive(self, start, visited=None) -> list:
        if visited is None: visited = set()
        visited.add(start)
        order = [start]
        for neighbor, _ in self.adj[start]:
            if neighbor not in visited:
                order.extend(self.dfs_recursive(neighbor, visited))
        return order

# Test BFS/DFS
g = Graph(directed=False)
for u, v in [('A','B'),('A','C'),('B','D'),('C','D'),('D','E'),('B','E')]:
    g.add_edge(u, v)

print("BFS distances from A:", g.bfs('A'))
print("Shortest path A→E:",    g.shortest_path_bfs('A', 'E'))
print("DFS order from A:",     g.dfs_iterative('A'))

In [ ]:
# ---- Dijkstra's Algorithm — Shortest path in weighted graph ----
# Time: O((V + E) log V), Space: O(V)

def dijkstra(graph: Graph, start) -> tuple:
    """Returns (distances, predecessors) from start."""
    dist = {node: float('inf') for node in graph.nodes()}
    prev = {node: None for node in graph.nodes()}
    dist[start] = 0

    # Min-heap: (distance, node)
    pq = [(0, start)]

    while pq:
        d, node = heapq.heappop(pq)
        if d > dist[node]:
            continue  # Stale entry in the heap

        for neighbor, weight in graph.adj[node]:
            new_dist = dist[node] + weight
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                prev[neighbor] = node
                heapq.heappush(pq, (new_dist, neighbor))

    return dist, prev

def reconstruct_path(prev, start, end) -> list:
    path = []
    node = end
    while node is not None:
        path.append(node)
        node = prev.get(node)
    path.reverse()
    return path if path[0] == start else []

# Build weighted graph
wg = Graph(directed=True)
edges = [('A','B',4),('A','C',2),('C','B',1),('B','D',5),
         ('C','D',8),('D','E',2),('B','E',6)]
for u, v, w in edges:
    wg.add_edge(u, v, w)

dist, prev = dijkstra(wg, 'A')
print("Dijkstra distances from A:", {k: v for k, v in dist.items() if v != float('inf')})
print("Shortest path A→E:", reconstruct_path(prev, 'A', 'E'))
print("Shortest path A→D:", reconstruct_path(prev, 'A', 'D'))

In [ ]:
# ---- Bellman-Ford — handles NEGATIVE weights, detects negative cycles ----
# Time: O(V * E), Space: O(V)

def bellman_ford(edges: list, nodes: list, start) -> tuple:
    """
    edges: list of (u, v, weight)
    Returns (distances, has_negative_cycle)
    """
    dist = {n: float('inf') for n in nodes}
    dist[start] = 0

    # Relax all edges |V|-1 times
    for _ in range(len(nodes) - 1):
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w

    # Check for negative cycles (one more relaxation pass)
    has_negative_cycle = False
    for u, v, w in edges:
        if dist[u] + w < dist[v]:
            has_negative_cycle = True
            break

    return dist, has_negative_cycle

nodes = ['A', 'B', 'C', 'D', 'E']
edges = [('A','B',4),('A','C',2),('C','B',-3),('B','D',5),('D','E',2)]
dist, neg_cycle = bellman_ford(edges, nodes, 'A')
print("Bellman-Ford distances:", {k:v for k,v in dist.items() if v != float('inf')})
print("Negative cycle:", neg_cycle)

# ---- Floyd-Warshall — ALL-PAIRS shortest paths ----
# Time: O(V^3), Space: O(V^2)

def floyd_warshall(n: int, edges: list) -> list:
    """
    n: number of nodes (0-indexed)
    edges: list of (u, v, weight)
    Returns n×n distance matrix.
    """
    INF = float('inf')
    dist = [[INF] * n for _ in range(n)]
    for i in range(n): dist[i][i] = 0
    for u, v, w in edges: dist[u][v] = w

    for k in range(n):          # Intermediate node
        for i in range(n):      # Source
            for j in range(n):  # Destination
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]

    return dist

# 4 nodes (0,1,2,3)
fw_edges = [(0,1,3),(0,3,7),(1,0,8),(1,2,2),(2,0,5),(2,3,1),(3,0,2)]
matrix = floyd_warshall(4, fw_edges)
print("\nFloyd-Warshall distance matrix:")
for row in matrix:
    print([x if x != float('inf') else '∞' for x in row])

In [ ]:
# ---- Topological Sort — for DAGs (task scheduling, build systems) ----

def topological_sort_kahn(adj: dict) -> list:
    """Kahn's algorithm: BFS-based topological sort. Detects cycles."""
    in_degree = defaultdict(int)
    nodes = set(adj.keys())
    for u, neighbors in adj.items():
        for v in neighbors:
            in_degree[v] += 1
            nodes.add(v)

    queue = deque([n for n in nodes if in_degree[n] == 0])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in adj.get(node, []):
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)

    if len(order) != len(nodes):
        raise ValueError("Graph has a cycle — topological sort impossible")
    return order

# Task dependency graph
tasks = {
    'install_python': [],
    'install_pip':    ['install_python'],
    'install_deps':   ['install_pip'],
    'run_tests':      ['install_deps'],
    'build':          ['install_deps'],
    'deploy':         ['run_tests', 'build'],
}

# Convert to adjacency list (dependency → dependent)
adj = defaultdict(list)
for task, deps in tasks.items():
    for dep in deps:
        adj[dep].append(task)

order = topological_sort_kahn(adj)
print("Build order:", ' → '.join(order))

# ---- Union-Find (Disjoint Set Union) — cycle detection, MST ----
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank   = [0] * n
        self.components = n

    def find(self, x):  # With path compression
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y) -> bool:  # Returns False if already connected (cycle!)
        px, py = self.find(x), self.find(y)
        if px == py: return False  # Cycle detected!
        if self.rank[px] < self.rank[py]: px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]: self.rank[px] += 1
        self.components -= 1
        return True

    def connected(self, x, y): return self.find(x) == self.find(y)

# Kruskal's MST using Union-Find
def kruskal_mst(n: int, edges: list) -> tuple:
    """Returns (MST edges, total weight). edges: [(weight, u, v)]"""
    uf = UnionFind(n)
    mst_edges = []
    total_weight = 0

    for weight, u, v in sorted(edges):  # Process in ascending weight order
        if uf.union(u, v):
            mst_edges.append((u, v, weight))
            total_weight += weight

    return mst_edges, total_weight

edges = [(4,0,1),(2,0,2),(3,0,3),(1,1,2),(5,1,3),(6,2,3)]
mst, weight = kruskal_mst(4, edges)
print(f"MST edges: {mst}, total weight: {weight}")

---
# 🌲 SECTION 13 — Advanced Tree Data Structures

In [ ]:
# ---- Binary Search Tree (BST) ----
from __future__ import annotations
from typing import Optional, Generator
from dataclasses import dataclass, field

@dataclass
class BSTNode:
    val: int
    left: Optional['BSTNode']  = field(default=None, repr=False)
    right: Optional['BSTNode'] = field(default=None, repr=False)

class BST:
    def __init__(self): self.root = None

    def insert(self, val: int) -> None:
        self.root = self._insert(self.root, val)

    def _insert(self, node, val):
        if node is None: return BSTNode(val)
        if val < node.val:   node.left  = self._insert(node.left, val)
        elif val > node.val: node.right = self._insert(node.right, val)
        return node

    def search(self, val: int) -> bool:
        node = self.root
        while node:
            if val == node.val: return True
            node = node.left if val < node.val else node.right
        return False

    def delete(self, val: int) -> None:
        self.root = self._delete(self.root, val)

    def _delete(self, node, val):
        if node is None: return None
        if val < node.val:
            node.left = self._delete(node.left, val)
        elif val > node.val:
            node.right = self._delete(node.right, val)
        else:
            if node.left is None:  return node.right
            if node.right is None: return node.left
            # Find in-order successor (smallest in right subtree)
            succ = node.right
            while succ.left: succ = succ.left
            node.val   = succ.val
            node.right = self._delete(node.right, succ.val)
        return node

    def inorder(self) -> Generator[int, None, None]:
        def _inorder(node):
            if node:
                yield from _inorder(node.left)
                yield node.val
                yield from _inorder(node.right)
        yield from _inorder(self.root)

    def height(self, node=None) -> int:
        if node is None: node = self.root
        if node is None: return 0
        return 1 + max(self.height(node.left), self.height(node.right))

bst = BST()
for v in [5, 3, 7, 1, 4, 6, 8, 2]:
    bst.insert(v)

print("Inorder:", list(bst.inorder()))  # Sorted!
print("Height:",  bst.height())
print("Search 4:", bst.search(4))
bst.delete(3)
print("After delete 3:", list(bst.inorder()))

In [ ]:
# ---- Trie (Prefix Tree) — O(L) insert/search, L = word length ----
# Use case: autocomplete, spell check, IP routing

class TrieNode:
    __slots__ = ('children', 'is_end', 'count')
    def __init__(self):
        self.children: dict[str, 'TrieNode'] = {}
        self.is_end = False
        self.count  = 0  # How many words pass through this node

class Trie:
    def __init__(self): self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
            node.count += 1
        node.is_end = True

    def search(self, word: str) -> bool:
        node = self._traverse(word)
        return node is not None and node.is_end

    def starts_with(self, prefix: str) -> bool:
        return self._traverse(prefix) is not None

    def _traverse(self, s: str) -> Optional[TrieNode]:
        node = self.root
        for char in s:
            if char not in node.children: return None
            node = node.children[char]
        return node

    def autocomplete(self, prefix: str) -> list[str]:
        """Return all words with given prefix."""
        node = self._traverse(prefix)
        if node is None: return []
        results = []
        self._dfs(node, list(prefix), results)
        return results

    def _dfs(self, node: TrieNode, path: list, results: list):
        if node.is_end: results.append(''.join(path))
        for char, child in sorted(node.children.items()):
            path.append(char)
            self._dfs(child, path, results)
            path.pop()

    def count_prefix(self, prefix: str) -> int:
        node = self._traverse(prefix)
        return node.count if node else 0

trie = Trie()
words = ['apple', 'app', 'application', 'apply', 'apt', 'banana', 'band', 'bandana']
for w in words: trie.insert(w)

print("search 'app':"   , trie.search('app'))      # True
print("search 'ap':"    , trie.search('ap'))        # False
print("starts_with 'ap':", trie.starts_with('ap'))  # True
print("autocomplete 'app':", trie.autocomplete('app'))
print("autocomplete 'ban':", trie.autocomplete('ban'))
print("count_prefix 'app':", trie.count_prefix('app'))

In [ ]:
# ---- Segment Tree — Range queries + point updates in O(log n) ----
# Use case: range sum/min/max, interval queries

class SegmentTree:
    def __init__(self, data: list):
        self.n = len(data)
        self.tree = [0] * (4 * self.n)
        self._build(data, 1, 0, self.n - 1)

    def _build(self, data, node, start, end):
        if start == end:
            self.tree[node] = data[start]
            return
        mid = (start + end) // 2
        self._build(data, 2*node,   start, mid)
        self._build(data, 2*node+1, mid+1, end)
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]

    def update(self, idx: int, val: int) -> None:
        self._update(1, 0, self.n-1, idx, val)

    def _update(self, node, start, end, idx, val):
        if start == end:
            self.tree[node] = val
            return
        mid = (start + end) // 2
        if idx <= mid: self._update(2*node, start, mid, idx, val)
        else:          self._update(2*node+1, mid+1, end, idx, val)
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]

    def query(self, l: int, r: int) -> int:
        """Sum of elements in range [l, r]."""
        return self._query(1, 0, self.n-1, l, r)

    def _query(self, node, start, end, l, r):
        if r < start or end < l: return 0          # Out of range
        if l <= start and end <= r: return self.tree[node]  # Fully covered
        mid = (start + end) // 2
        return (self._query(2*node, start, mid, l, r) +
                self._query(2*node+1, mid+1, end, l, r))

arr = [1, 3, 5, 7, 9, 11]
st  = SegmentTree(arr)

print(f"Sum [1,3]:  {st.query(1, 3)}")  # 3+5+7 = 15
print(f"Sum [0,5]:  {st.query(0, 5)}")  # 36

st.update(2, 10)  # arr[2] = 10 (was 5)
print(f"Sum [1,3] after update: {st.query(1, 3)}")  # 3+10+7 = 20

---
# ⚙️ SECTION 14 — Advanced Sorting, Searching & Bit Manipulation

In [ ]:
import random

# ---- Quick Sort — O(n log n) avg, O(n²) worst, O(log n) space ----
def quicksort(arr: list, lo=0, hi=None) -> list:
    if hi is None: arr, lo, hi = arr[:], 0, len(arr)-1

    def partition(lo, hi):
        # Median-of-three pivot for better average performance
        mid = (lo + hi) // 2
        pivot_candidates = [(arr[lo], lo), (arr[mid], mid), (arr[hi], hi)]
        pivot_val, pivot_idx = sorted(pivot_candidates)[1]
        arr[pivot_idx], arr[hi] = arr[hi], arr[pivot_idx]

        pivot = arr[hi]
        i = lo - 1
        for j in range(lo, hi):
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i+1], arr[hi] = arr[hi], arr[i+1]
        return i + 1

    def _sort(lo, hi):
        if lo < hi:
            p = partition(lo, hi)
            _sort(lo, p-1)
            _sort(p+1, hi)

    _sort(lo, hi)
    return arr

# ---- Merge Sort — O(n log n) always, O(n) space ----
def mergesort(arr: list) -> list:
    if len(arr) <= 1: return arr
    mid   = len(arr) // 2
    left  = mergesort(arr[:mid])
    right = mergesort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]); i += 1
        else:                    result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

# ---- Counting Sort — O(n+k) for integers ----
def counting_sort(arr: list, max_val: int = None) -> list:
    if not arr: return arr
    if max_val is None: max_val = max(arr)
    count = [0] * (max_val + 1)
    for x in arr: count[x] += 1
    result = []
    for val, cnt in enumerate(count):
        result.extend([val] * cnt)
    return result

# Test all sorts
test_arr = random.sample(range(100), 20)
print("Original:",  test_arr)
print("Quicksort:", quicksort(test_arr.copy()))
print("Mergesort:", mergesort(test_arr))
print("Counting:",  counting_sort(test_arr))

In [ ]:
# ---- Binary Search Variants ----

def binary_search(arr, target) -> int:
    """Standard: exact match. Returns index or -1."""
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2  # Avoids overflow
        if arr[mid] == target: return mid
        elif arr[mid] < target: lo = mid + 1
        else: hi = mid - 1
    return -1

def lower_bound(arr, target) -> int:
    """First index where arr[i] >= target (like bisect_left)."""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < target: lo = mid + 1
        else: hi = mid
    return lo

def upper_bound(arr, target) -> int:
    """First index where arr[i] > target (like bisect_right)."""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] <= target: lo = mid + 1
        else: hi = mid
    return lo

def search_rotated(arr, target) -> int:
    """Binary search in rotated sorted array."""
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target: return mid
        if arr[lo] <= arr[mid]:   # Left half is sorted
            if arr[lo] <= target < arr[mid]: hi = mid - 1
            else: lo = mid + 1
        else:                     # Right half is sorted
            if arr[mid] < target <= arr[hi]: lo = mid + 1
            else: hi = mid - 1
    return -1

arr = [1, 2, 2, 3, 3, 3, 4, 5]
print(f"binary_search(3):  {binary_search(arr, 3)}")  # Any index of 3
print(f"lower_bound(3):    {lower_bound(arr, 3)}")  # 3 — first index of 3
print(f"upper_bound(3):    {upper_bound(arr, 3)}")  # 6 — past last 3

rotated = [4, 5, 6, 7, 0, 1, 2]
print(f"search_rotated(0): {search_rotated(rotated, 0)}")  # 4

In [ ]:
# ---- Bit Manipulation ----

# Fundamental operations
def bit_tricks():
    n = 0b1010_1100  # 172

    print(f"n = {n} = {bin(n)}")
    print(f"Check bit 3:     {bool(n & (1 << 3))}")     # True (bit 3 is set)
    print(f"Set bit 1:       {bin(n | (1 << 1))}")      # Set bit 1
    print(f"Clear bit 3:     {bin(n & ~(1 << 3))}")     # Clear bit 3
    print(f"Toggle bit 1:    {bin(n ^ (1 << 1))}")      # Toggle bit 1
    print(f"Clear lowest set bit: {bin(n & (n-1))}")    # n & (n-1) magic
    print(f"Isolate lowest set bit: {bin(n & (-n))}")   # n & (-n) magic
    print(f"Is power of 2 (64): {64 & 63 == 0 and 64 != 0}")  # True
    print(f"Count set bits: {bin(n).count('1')}")       # popcount
    print(f"popcount (builtin): {n.bit_count()}")       # Python 3.10+

bit_tricks()

# ---- Classic Bit DP problems ----

# Find the single number (all others appear twice) — XOR trick
def single_number(nums: list) -> int:
    result = 0
    for n in nums: result ^= n  # XOR cancels pairs
    return result

print(f"\nSingle number: {single_number([4, 1, 2, 1, 2])}")  # 4

# Hamming distance
def hamming_distance(x: int, y: int) -> int:
    return (x ^ y).bit_count()  # Count differing bits

print(f"Hamming(1,4): {hamming_distance(1, 4)}")  # 2

# Bitmask DP — Travelling Salesman Problem subset DP
def tsp_bitmask(dist: list) -> int:
    n = len(dist)
    # dp[mask][i] = min cost to visit all cities in mask, ending at i
    INF = float('inf')
    dp = [[INF] * n for _ in range(1 << n)]
    dp[1][0] = 0  # Start at city 0, only city 0 visited

    for mask in range(1 << n):
        for u in range(n):
            if dp[mask][u] == INF: continue
            if not (mask >> u & 1): continue  # u not in mask
            for v in range(n):
                if mask >> v & 1: continue    # v already visited
                new_mask = mask | (1 << v)
                dp[new_mask][v] = min(dp[new_mask][v], dp[mask][u] + dist[u][v])

    full_mask = (1 << n) - 1
    return min(dp[full_mask][i] + dist[i][0] for i in range(n))

# 4 cities
d = [[0,10,15,20],[10,0,35,25],[15,35,0,30],[20,25,30,0]]
print(f"TSP min cost: {tsp_bitmask(d)}")  # 80

---
# 🧪 SECTION 15 — Testing with pytest

In [ ]:
# ---- Writing pytest-style tests inline ----
# In a real project: save as test_*.py and run: pytest -v

# ---- Module under test ----
class Stack:
    def __init__(self): self._items = []
    def push(self, item): self._items.append(item)
    def pop(self):
        if not self._items: raise IndexError("Stack is empty")
        return self._items.pop()
    def peek(self):
        if not self._items: raise IndexError("Stack is empty")
        return self._items[-1]
    def __len__(self): return len(self._items)
    def __bool__(self): return bool(self._items)

# ---- Tests ----
import pytest

class TestStack:
    def test_push_and_len(self):
        s = Stack()
        s.push(1)
        s.push(2)
        assert len(s) == 2

    def test_pop_returns_last(self):
        s = Stack()
        s.push(10)
        s.push(20)
        assert s.pop() == 20
        assert len(s) == 1

    def test_pop_empty_raises(self):
        s = Stack()
        with pytest.raises(IndexError, match='Stack is empty'):
            s.pop()

    def test_peek_does_not_remove(self):
        s = Stack()
        s.push(42)
        assert s.peek() == 42
        assert len(s) == 1

    def test_bool_empty(self):
        assert not Stack()

    def test_bool_non_empty(self):
        s = Stack()
        s.push(None)  # Even None is a valid item
        assert s

# Run tests manually (in Jupyter)
import subprocess, sys

# Save tests to a temporary file and run pytest
test_code = '''
class Stack:
    def __init__(self): self._items = []
    def push(self, item): self._items.append(item)
    def pop(self):
        if not self._items: raise IndexError("Stack is empty")
        return self._items.pop()
    def peek(self):
        if not self._items: raise IndexError("Stack is empty")
        return self._items[-1]
    def __len__(self): return len(self._items)
    def __bool__(self): return bool(self._items)

import pytest

def test_push(): s=Stack(); s.push(1); assert len(s)==1
def test_pop():  s=Stack(); s.push(1); assert s.pop()==1
def test_raises(): 
    s=Stack()
    with pytest.raises(IndexError): s.pop()
'''

with open('/tmp/test_stack.py', 'w') as f:
    f.write(test_code)

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '/tmp/test_stack.py', '-v', '--tb=short'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])

In [ ]:
# ---- Fixtures, Parametrize, Mocking ----

advanced_tests = '''
import pytest
from unittest.mock import Mock, patch, MagicMock, call

# ---- FIXTURE ----
@pytest.fixture
def sample_data():
    """Reusable test data."""
    return [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]

@pytest.fixture
def db_connection():
    """Setup/teardown fixture."""
    conn = {"connected": True, "data": {}}
    yield conn                # Test code runs here
    conn["connected"] = False  # Teardown

def test_sort_fixture(sample_data):
    assert sorted(sample_data) == [1,1,2,3,3,4,5,5,6,9]

def test_db_fixture(db_connection):
    assert db_connection["connected"]

# ---- PARAMETRIZE ----
@pytest.mark.parametrize("n, expected", [
    (0, 0), (1, 1), (5, 5), (10, 55), (20, 6765)
])
def test_fibonacci(n, expected):
    from functools import lru_cache
    @lru_cache
    def fib(n): return n if n <= 1 else fib(n-1)+fib(n-2)
    assert fib(n) == expected

@pytest.mark.parametrize("a,b,op,result", [
    (2,3,"add",5), (10,3,"sub",7), (4,5,"mul",20), (10,2,"div",5.0)
])
def test_calculator(a, b, op, result):
    ops = {"add":lambda a,b:a+b, "sub":lambda a,b:a-b,
           "mul":lambda a,b:a*b, "div":lambda a,b:a/b}
    assert ops[op](a, b) == result

# ---- MOCKING ----
class EmailService:
    def send(self, to, subject, body): pass  # External dependency

class UserRegistration:
    def __init__(self, email_service: EmailService):
        self.email_service = email_service

    def register(self, email, name):
        # ... save to DB ...
        self.email_service.send(
            to=email,
            subject="Welcome!",
            body=f"Hi {name}, welcome!"
        )
        return True

def test_registration_sends_email():
    mock_email = Mock(spec=EmailService)
    reg = UserRegistration(mock_email)

    result = reg.register("alice@example.com", "Alice")

    assert result is True
    mock_email.send.assert_called_once_with(
        to="alice@example.com",
        subject="Welcome!",
        body="Hi Alice, welcome!"
    )

def test_registration_multiple_calls():
    mock_email = Mock()
    reg = UserRegistration(mock_email)
    reg.register("a@a.com", "A")
    reg.register("b@b.com", "B")
    assert mock_email.send.call_count == 2

# patch as context manager
def test_with_patch():
    with patch("builtins.open", mock_open := MagicMock()):
        mock_open.return_value.__enter__.return_value.read.return_value = "data"
        # test code that calls open()
'''

with open('/tmp/test_advanced.py', 'w') as f:
    f.write(advanced_tests)

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '/tmp/test_advanced.py', '-v', '--tb=short'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])

---
# 🔍 SECTION 16 — Regular Expressions (Expert Level)

In [ ]:
import re

# ---- Flags ----
text = "Hello World\nPython is GREAT"
print(re.findall(r'\w+', text, re.IGNORECASE | re.MULTILINE))

# ---- Named groups ----
log = "2024-01-15 ERROR: Connection failed to 192.168.1.1:5432"
pattern = r'(?P<date>\d{4}-\d{2}-\d{2}) (?P<level>\w+): (?P<msg>.+)'
m = re.match(pattern, log)
if m:
    print(m.group('date'))   # 2024-01-15
    print(m.group('level'))  # ERROR
    print(m.groupdict())     # All named groups as dict

# ---- Lookahead & Lookbehind ----
prices = "apple: $1.50, banana: $0.75, cherry: $3.00"

# Positive lookahead: match digits followed by .00
whole = re.findall(r'\$\d+\.00', prices)
print("Whole dollar prices:", whole)  # ['$3.00']

# Positive lookbehind: find prices preceded by '$'
amounts = re.findall(r'(?<=\$)[\d.]+', prices)
print("Amounts:", amounts)  # ['1.50', '0.75', '3.00']

# Negative lookbehind: words NOT preceded by 'un'
text2 = "happy unhappy common uncommon"
words = re.findall(r'(?<!un)\b\w+', text2)
print("Not 'un' words:", words)

# ---- Non-capturing groups (?:) ----
# (?:...) groups without capturing
dates = re.findall(r'(?:\d{4})-(?:\d{2})-(?:\d{2})', "2024-01-15 and 2023-12-31")
print("Dates:", dates)

# ---- Atomic groups / possessive quantifiers (Python 3.11+) ----
# (?>) prevents backtracking inside group

In [ ]:
# ---- Substitution with function ----
def camel_to_snake(name: str) -> str:
    s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

tests = ['CamelCase', 'HTTPSRequest', 'XMLParser', 'getHTTPSUrl', 'myVariableName']
for t in tests:
    print(f"{t:20s} → {camel_to_snake(t)}")

# ---- re.sub with callable ----
template = "Hello {NAME}, your order #{ORDER_ID} is ready!"
values   = {'NAME': 'Alice', 'ORDER_ID': '12345'}

result = re.sub(r'\{(\w+)\}', lambda m: values.get(m.group(1), m.group(0)), template)
print(result)  # Hello Alice, your order #12345 is ready!

# ---- Compiled patterns — performance ----
# Compile once, use many times
EMAIL_RE   = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')
IPV4_RE    = re.compile(r'^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$')
URL_RE     = re.compile(r'https?://(?:www\.)?[-a-zA-Z0-9@:%._+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b[-a-zA-Z0-9@:%_+.~#?&/=]*')

for email in ['user@example.com', 'bad@', 'ok@domain.co.uk']:
    print(f"{email}: {bool(EMAIL_RE.match(email))}")

for ip in ['192.168.1.1', '256.0.0.1', '10.0.0.1']:
    print(f"{ip}: {bool(IPV4_RE.match(ip))}")

# ---- Verbose regex for readability ----
DATE_RE = re.compile(r'''
    (?P<year>  \d{4}) -    # Year: 4 digits
    (?P<month> \d{2}) -    # Month: 2 digits
    (?P<day>   \d{2})      # Day: 2 digits
''', re.VERBOSE)

m = DATE_RE.match('2024-01-15')
print(m.groupdict())  # {'year': '2024', 'month': '01', 'day': '15'}

---
# 📁 SECTION 17 — File I/O, Pathlib & Serialization

In [ ]:
from pathlib import Path
import os

# ---- Pathlib — modern, cross-platform file paths ----
# Prefer pathlib over os.path for new code.

home = Path.home()
cwd  = Path.cwd()

# Path construction
config_dir = home / '.config' / 'myapp'  # Works on all OSes!
data_file  = Path('/tmp/data.json')

print(f"Home:      {home}")
print(f"CWD:       {cwd}")
print(f"Config:    {config_dir}")
print(f"Parent:    {data_file.parent}")
print(f"Name:      {data_file.name}")
print(f"Stem:      {data_file.stem}")
print(f"Suffix:    {data_file.suffix}")
print(f"Absolute:  {data_file.is_absolute()}")

# Path operations
new_path = data_file.with_suffix('.csv').with_stem('output')
print(f"Renamed:   {new_path}")

# Create directories
tmp_dir = Path('/tmp/myapp_test')
tmp_dir.mkdir(parents=True, exist_ok=True)  # No error if exists

# Write and read
test_file = tmp_dir / 'test.txt'
test_file.write_text('Hello, Pathlib!\nLine 2')
content = test_file.read_text()
print(f"\nRead back: {content!r}")

# Glob
for f in Path('/tmp').glob('*.txt'):
    print(f"Found: {f.name}")

# Recursive glob
# for py_file in Path('.').rglob('*.py'):
#     print(py_file)

# File info
stat = test_file.stat()
print(f"Size: {stat.st_size} bytes")

# Cleanup
test_file.unlink()  # Delete file
tmp_dir.rmdir()     # Delete empty dir

In [ ]:
import json
import pickle
import csv
import io
from dataclasses import dataclass, asdict

# ---- JSON ----
@dataclass
class User:
    name: str
    age:  int
    tags: list

users = [User('Alice', 30, ['admin']), User('Bob', 25, ['user', 'viewer'])]

# Custom encoder for dataclasses
class DataclassEncoder(json.JSONEncoder):
    def default(self, obj):
        if hasattr(obj, '__dataclass_fields__'):
            return asdict(obj)
        return super().default(obj)

json_str = json.dumps(users, cls=DataclassEncoder, indent=2)
print(json_str)

# Decode with object_hook
def user_decoder(d: dict):
    if 'name' in d and 'age' in d and 'tags' in d:
        return User(**d)
    return d

loaded = json.loads(json_str, object_hook=user_decoder)
print(f"Loaded: {loaded}")

# ---- Pickle — serialize ANY Python object ----
# WARNING: Pickle is NOT secure — never unpickle untrusted data!

data = {'model': users, 'meta': {'version': 1}}

pickled = pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL)
print(f"\nPickled size: {len(pickled)} bytes")

restored = pickle.loads(pickled)
print(f"Restored: {restored['model']}")

# Custom pickling for classes
class SecureData:
    def __init__(self, value):
        self.value = value
        self._secret = 'dont_pickle_me'

    def __getstate__(self):
        state = self.__dict__.copy()
        del state['_secret']  # Exclude sensitive data from pickle
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._secret = 'restored_default'  # Restore with default

sd = SecureData(42)
sd2 = pickle.loads(pickle.dumps(sd))
print(f"Value: {sd2.value}, Secret: {sd2._secret}")

# ---- CSV ----
csv_data = io.StringIO()
writer = csv.DictWriter(csv_data, fieldnames=['name', 'age', 'tags'])
writer.writeheader()
for u in users:
    writer.writerow({'name': u.name, 'age': u.age, 'tags': ','.join(u.tags)})

print("\nCSV output:")
print(csv_data.getvalue())

---
# 📋 SECTION 18 — Logging (Production Grade)

In [ ]:
import logging
import logging.handlers
import json
import sys
from datetime import datetime

# ---- Logging hierarchy ----
# root → app → app.module → app.module.submodule
# Each logger inherits handlers from parent unless propagate=False

# ---- Structured JSON logging ----
class JSONFormatter(logging.Formatter):
    """Outputs log records as JSON — great for log aggregation (ELK, Datadog)."""

    def format(self, record: logging.LogRecord) -> str:
        log_data = {
            'timestamp': datetime.utcfromtimestamp(record.created).isoformat() + 'Z',
            'level':     record.levelname,
            'logger':    record.name,
            'message':   record.getMessage(),
            'module':    record.module,
            'func':      record.funcName,
            'line':      record.lineno,
        }
        if record.exc_info:
            log_data['exception'] = self.formatException(record.exc_info)
        if hasattr(record, 'extra'):
            log_data.update(record.extra)
        return json.dumps(log_data)

# ---- Setup logger ----
def setup_logger(name: str, level=logging.DEBUG) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers.clear()  # Remove existing handlers

    # Console handler — human-readable
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG)
    console_handler.setFormatter(logging.Formatter(
        '%(asctime)s [%(levelname)-8s] %(name)s: %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))

    # Rotating file handler — max 5MB, keep 3 backups
    file_handler = logging.handlers.RotatingFileHandler(
        '/tmp/app.log', maxBytes=5*1024*1024, backupCount=3
    )
    file_handler.setLevel(logging.WARNING)
    file_handler.setFormatter(JSONFormatter())

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    logger.propagate = False  # Don't bubble to root

    return logger

app_log = setup_logger('myapp')
db_log  = setup_logger('myapp.database')

app_log.info("Application starting")
app_log.debug("Debug: config loaded")
app_log.warning("Disk usage > 80%%")

try:
    raise ConnectionError("DB host unreachable")
except ConnectionError:
    db_log.error("Database connection failed", exc_info=True)

# ---- LoggerAdapter — add context to every message ----
class RequestAdapter(logging.LoggerAdapter):
    def process(self, msg, kwargs):
        return f"[req:{self.extra['request_id']}] {msg}", kwargs

request_log = RequestAdapter(app_log, {'request_id': 'abc-123'})
request_log.info("Processing request")  # [req:abc-123] Processing request
request_log.info("Request complete")

---
# 🌐 SECTION 19 — Networking & Sockets

In [ ]:
import socket
import threading
import time

# ---- TCP Server & Client ----

def run_echo_server(host='127.0.0.1', port=12345):
    """Simple echo server: sends back whatever it receives."""
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((host, port))
    server.listen(1)
    server.settimeout(2.0)  # Don't block forever

    try:
        conn, addr = server.accept()
        print(f"Server: connected by {addr}")
        with conn:
            while True:
                data = conn.recv(1024)
                if not data: break
                print(f"Server received: {data.decode()!r}")
                conn.sendall(b'Echo: ' + data)
    except socket.timeout:
        print("Server: timeout, no connection")
    finally:
        server.close()

def run_client(host='127.0.0.1', port=12345, messages=None):
    time.sleep(0.1)  # Wait for server to start
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client:
        client.connect((host, port))
        for msg in (messages or ['Hello!', 'How are you?']):
            client.sendall(msg.encode())
            response = client.recv(1024)
            print(f"Client received: {response.decode()!r}")

# Run server in background thread
server_thread = threading.Thread(target=run_echo_server, daemon=True)
server_thread.start()
run_client()
server_thread.join(timeout=3)

In [ ]:
# ---- Async TCP Server with asyncio.start_server ----
import asyncio

async def async_echo_demo():
    # Handler for each client connection
    async def handle_client(reader: asyncio.StreamReader,
                            writer: asyncio.StreamWriter):
        addr = writer.get_extra_info('peername')
        print(f"Async server: new connection from {addr}")

        try:
            while True:
                data = await reader.read(1024)
                if not data: break
                msg = data.decode().strip()
                print(f"Async server received: {msg!r}")
                writer.write(f"Echo: {msg}\n".encode())
                await writer.drain()  # Flush the buffer
        finally:
            writer.close()
            await writer.wait_closed()

    # Start server
    server = await asyncio.start_server(handle_client, '127.0.0.1', 12346)

    async def async_client():
        await asyncio.sleep(0.1)
        reader, writer = await asyncio.open_connection('127.0.0.1', 12346)
        for msg in ['Hello async!', 'Goodbye!']:
            writer.write(msg.encode())
            await writer.drain()
            response = await reader.readline()
            print(f"Async client received: {response.decode().strip()!r}")
        writer.close()
        await writer.wait_closed()
        server.close()

    async with server:
        await async_client()

await async_echo_demo()

# ---- UDP socket ----
def udp_demo():
    # UDP: connectionless, no guarantee of delivery
    # Sender
    sender = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

    # Receiver
    receiver = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    receiver.bind(('127.0.0.1', 12347))
    receiver.settimeout(1.0)

    sender.sendto(b'UDP message!', ('127.0.0.1', 12347))
    try:
        data, addr = receiver.recvfrom(1024)
        print(f"UDP received: {data.decode()!r} from {addr}")
    except socket.timeout:
        print("UDP: timeout")

    sender.close()
    receiver.close()

udp_demo()

---
# 🏷️ SECTION 20 — Advanced Type System

In [ ]:
from typing import (
    TypeVar, Generic, Protocol, overload, TypeGuard,
    Callable, Iterator, Generator, Awaitable,
    NamedTuple, TypedDict, Annotated, Literal, Final,
    get_type_hints, runtime_checkable
)
from typing import ParamSpec, Concatenate  # Python 3.10+

T = TypeVar('T')
T_co = TypeVar('T_co', covariant=True)      # Covariant — producer
T_contra = TypeVar('T_contra', contravariant=True)  # Contravariant — consumer
P = ParamSpec('P')  # Captures parameter spec

# ---- Generic Classes ----
class Stack(Generic[T]):
    """Type-safe stack."""
    def __init__(self): self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

    def peek(self) -> T:
        return self._items[-1]

# Type checker understands these:
int_stack: Stack[int]  = Stack()
str_stack: Stack[str]  = Stack()
int_stack.push(42)     # OK
# int_stack.push("hi") # Type error (mypy would catch this)

# ---- Bounded TypeVar ----
Comparable = TypeVar('Comparable', int, float, str)  # Constraint: only these types
NumberT    = TypeVar('NumberT', bound='Number')

def maximum(items: list[Comparable]) -> Comparable:
    return max(items)

print(maximum([1, 5, 3, 2]))       # 5
print(maximum(['z', 'a', 'm']))    # z

# ---- @overload — multiple signatures ----
@overload
def process(value: int) -> str: ...
@overload
def process(value: str) -> int: ...
@overload
def process(value: list) -> dict: ...

def process(value):
    if isinstance(value, int):  return str(value)
    if isinstance(value, str):  return len(value)
    if isinstance(value, list): return {i: v for i, v in enumerate(value)}
    raise TypeError(f"Unsupported: {type(value)}")

print(process(42))          # '42'
print(process('hello'))     # 5
print(process(['a','b']))   # {0:'a', 1:'b'}

In [ ]:
# ---- TypeGuard — narrow types in conditionals ----
def is_list_of_str(val: list) -> TypeGuard[list[str]]:
    return all(isinstance(x, str) for x in val)

def process_strings(values: list) -> None:
    if is_list_of_str(values):
        # Type narrowed: values is list[str] here
        print(', '.join(values))

process_strings(['a', 'b', 'c'])

# ---- Annotated — attach metadata to types ----
from typing import Annotated

# Can attach validators, documentation, constraints
PositiveInt  = Annotated[int, 'must be > 0']
EmailAddress = Annotated[str, re.compile(r'^[\w.]+@[\w]+\.[\w]+$')]

def create_user(name: str, age: PositiveInt, email: EmailAddress) -> dict:
    return {'name': name, 'age': age, 'email': email}

# ---- Literal — restrict to specific values ----
Direction = Literal['north', 'south', 'east', 'west']
LogLevel  = Literal['DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL']

def move(direction: Direction, steps: int) -> str:
    return f"Moving {steps} steps {direction}"

print(move('north', 5))
# move('up', 5)  # Type error — 'up' not in Literal

# ---- Final — constants ----
MAX_RETRIES: Final = 3
API_VERSION: Final[str] = 'v2'
# MAX_RETRIES = 5  # Type error — can't reassign Final

# ---- ParamSpec — preserve callable signatures ----
import functools

def add_logging(func: Callable[P, T]) -> Callable[P, T]:
    """Decorator that preserves the complete type signature."""
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Done {func.__name__}")
        return result
    return wrapper

@add_logging
def add(x: int, y: int) -> int:
    return x + y

print(add(3, 4))  # Type checker knows return type is int

---
# 🎯 SECTION 21 — Structural Pattern Matching (Python 3.10+)

`match/case` is not a `switch` statement — it's full structural pattern matching.

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:   x: float; y: float

@dataclass
class Circle:  center: Point; radius: float

@dataclass
class Rectangle: top_left: Point; bottom_right: Point

Shape = Circle | Rectangle | Point

def describe_shape(shape: Shape) -> str:
    match shape:
        # Class patterns — destructure dataclasses
        case Circle(center=Point(x=0, y=0), radius=r):
            return f"Circle centered at origin, radius={r}"

        case Circle(center=c, radius=r) if r > 100:
            return f"Large circle at {c}, radius={r}"

        case Circle(center=c, radius=r):
            return f"Circle at ({c.x},{c.y}), radius={r}"

        case Rectangle(top_left=Point(x=x1,y=y1), bottom_right=Point(x=x2,y=y2)):
            w, h = abs(x2-x1), abs(y2-y1)
            return f"Rectangle {w}x{h} at ({x1},{y1})"

        case Point(x=x, y=y):
            return f"Point at ({x},{y})"

        case _:
            return "Unknown shape"

shapes = [
    Circle(Point(0,0), 5),
    Circle(Point(1,2), 200),
    Circle(Point(3,4), 10),
    Rectangle(Point(0,10), Point(5,0)),
    Point(1,2),
]
for s in shapes:
    print(describe_shape(s))

In [ ]:
# ---- Sequence, mapping and OR patterns ----

def process_command(command: list | dict | str) -> str:
    match command:
        # Sequence patterns
        case ['quit'] | ['exit']:
            return "Exiting..."

        case ['go', direction] if direction in ('north','south','east','west'):
            return f"Going {direction}"

        case ['go', direction]:
            return f"Unknown direction: {direction}"

        case ['get', *items]:  # *items captures remaining
            return f"Getting: {items}"

        case [first, *rest] if len(rest) > 3:
            return f"Long command: {first!r} followed by {len(rest)} args"

        # Mapping patterns
        case {'action': 'move', 'direction': d, 'speed': s}:
            return f"Moving {d} at speed {s}"

        case {'action': action, **rest}:  # **rest captures remaining keys
            return f"Action {action!r} with extra params: {rest}"

        # Literal patterns
        case 'help':
            return "Available commands: go, get, quit"

        case str() as s:
            return f"Unknown string command: {s!r}"

        case _:
            return f"Unknown command: {command!r}"

commands = [
    ['quit'],
    ['go', 'north'],
    ['go', 'up'],
    ['get', 'sword', 'shield', 'potion'],
    {'action': 'move', 'direction': 'left', 'speed': 5},
    {'action': 'attack', 'target': 'dragon', 'power': 100},
    'help',
]

for cmd in commands:
    print(f"{str(cmd):50s} → {process_command(cmd)}")

---
# 🧬 SECTION 22 — Advanced OOP: Mixins, Class Factories & Protocols

In [ ]:
# ---- Mixins — reusable behaviour without inheritance hierarchy ----
# Mixins: small, focused classes meant to be mixed in to other classes.
# They should NOT have __init__ with meaningful state.

class SerializeMixin:
    """Adds JSON serialization to any class with a __dict__."""
    def to_json(self) -> str:
        import json
        return json.dumps(self.__dict__, default=str)

    @classmethod
    def from_json(cls, json_str: str):
        import json
        data = json.loads(json_str)
        obj = cls.__new__(cls)
        obj.__dict__.update(data)
        return obj

class ValidateMixin:
    """Adds validation hook."""
    _validators: dict = {}

    def validate(self) -> list[str]:
        errors = []
        for field, validator in self._validators.items():
            value = getattr(self, field, None)
            if not validator(value):
                errors.append(f"Validation failed for {field}={value!r}")
        return errors

class LogMixin:
    """Adds a class-specific logger."""
    import logging
    @property
    def log(self):
        import logging
        return logging.getLogger(type(self).__name__)

class User(SerializeMixin, ValidateMixin, LogMixin):
    _validators = {
        'name':  lambda v: isinstance(v, str) and len(v) > 0,
        'age':   lambda v: isinstance(v, int) and 0 < v < 150,
        'email': lambda v: v and '@' in v,
    }

    def __init__(self, name, age, email):
        self.name  = name
        self.age   = age
        self.email = email

u = User('Alice', 30, 'alice@example.com')
print("JSON:", u.to_json())
print("Errors:", u.validate())

bad_user = User('', -1, 'not-an-email')
print("Bad errors:", bad_user.validate())

restored = User.from_json(u.to_json())
print("Restored:", restored.name, restored.age)

In [ ]:
# ---- Class Factories — dynamic class generation ----

def make_validator_class(name: str, fields: dict) -> type:
    """
    Dynamically creates a validated dataclass.
    fields: {'field_name': (type, validator_fn)}
    """
    def __init__(self, **kwargs):
        for field_name, (field_type, _) in fields.items():
            value = kwargs.get(field_name)
            if not isinstance(value, field_type):
                raise TypeError(f"{field_name} must be {field_type.__name__}")
            setattr(self, field_name, value)

    def validate(self):
        errors = []
        for field_name, (_, validator) in fields.items():
            if not validator(getattr(self, field_name)):
                errors.append(f"{field_name} failed validation")
        return errors

    def __repr__(self):
        attrs = ', '.join(f"{k}={getattr(self, k)!r}" for k in fields)
        return f"{name}({attrs})"

    return type(name, (), {
        '__init__': __init__,
        'validate': validate,
        '__repr__': __repr__,
    })

Product = make_validator_class('Product', {
    'name':  (str,   lambda v: len(v) > 0),
    'price': (float, lambda v: v > 0),
    'stock': (int,   lambda v: v >= 0),
})

p = Product(name='Widget', price=9.99, stock=100)
print(p)
print(p.validate())

# ---- Protocol with runtime checking ----
from typing import Protocol, runtime_checkable

@runtime_checkable
class Renderable(Protocol):
    def render(self) -> str: ...
    def dimensions(self) -> tuple[int, int]: ...

class HTMLButton:
    def render(self) -> str: return '<button>Click</button>'
    def dimensions(self) -> tuple[int, int]: return (100, 30)

class PDFBox:
    def render(self) -> str: return 'PDF box'
    def dimensions(self) -> tuple[int, int]: return (200, 50)

def render_all(items: list[Renderable]):
    for item in items:
        w, h = item.dimensions()
        print(f"[{w}x{h}] {item.render()}")

components = [HTMLButton(), PDFBox()]
render_all(components)

# Runtime check — no inheritance needed!
print(isinstance(HTMLButton(), Renderable))  # True
print(isinstance("not renderable", Renderable))  # False

---
# 🏛️ SECTION 23 — More Design Patterns

In [ ]:
# ════════════════════════════════════════════════
# VISITOR — add operations to objects without modifying them
# ════════════════════════════════════════════════
from abc import ABC, abstractmethod

# AST-like expression tree
class Expr(ABC):
    @abstractmethod
    def accept(self, visitor: 'ExprVisitor'): ...

class Num(Expr):
    def __init__(self, val): self.val = val
    def accept(self, v): return v.visit_num(self)

class Add(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def accept(self, v): return v.visit_add(self)

class Mul(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def accept(self, v): return v.visit_mul(self)

class ExprVisitor(ABC):
    @abstractmethod
    def visit_num(self, node: Num): ...
    @abstractmethod
    def visit_add(self, node: Add): ...
    @abstractmethod
    def visit_mul(self, node: Mul): ...

class EvalVisitor(ExprVisitor):
    def visit_num(self, node): return node.val
    def visit_add(self, node): return node.left.accept(self) + node.right.accept(self)
    def visit_mul(self, node): return node.left.accept(self) * node.right.accept(self)

class PrintVisitor(ExprVisitor):
    def visit_num(self, node): return str(node.val)
    def visit_add(self, node): return f"({node.left.accept(self)} + {node.right.accept(self)})"
    def visit_mul(self, node): return f"({node.left.accept(self)} * {node.right.accept(self)})"

# (2 + 3) * (4 + 1)
expr = Mul(Add(Num(2), Num(3)), Add(Num(4), Num(1)))

evaluator = EvalVisitor()
printer   = PrintVisitor()

print(f"Expression: {expr.accept(printer)}")   # (2+3)*(4+1)
print(f"Result:     {expr.accept(evaluator)}")  # 25

# ════════════════════════════════════════════════
# NULL OBJECT — eliminate None checks
# ════════════════════════════════════════════════
class Logger(ABC):
    @abstractmethod
    def log(self, msg): ...

class FileLogger(Logger):
    def __init__(self, path): self.path = path
    def log(self, msg): print(f"[FILE:{self.path}] {msg}")

class NullLogger(Logger):
    """Does nothing — no-op logger. Use instead of None checks."""
    def log(self, msg): pass  # Silent

class Service:
    def __init__(self, logger: Logger = None):
        self.logger = logger or NullLogger()  # Never None

    def do_work(self):
        self.logger.log("Starting work")  # Always safe — no None check
        return 42

s1 = Service(FileLogger('app.log'))
s1.do_work()  # Logs to file

s2 = Service()  # No logger
s2.do_work()  # Silent, no errors, no None checks

In [ ]:
# ════════════════════════════════════════════════
# MEDIATOR — decouple components via central coordinator
# ════════════════════════════════════════════════

class ChatMediator:
    """Central hub: components talk through mediator, not directly."""
    def __init__(self):
        self._users: dict[str, 'ChatUser'] = {}

    def register(self, user: 'ChatUser') -> None:
        self._users[user.name] = user

    def send(self, sender: str, recipient: str, msg: str) -> None:
        if recipient == 'all':
            for name, user in self._users.items():
                if name != sender:
                    user.receive(sender, msg)
        elif recipient in self._users:
            self._users[recipient].receive(sender, msg)

class ChatUser:
    def __init__(self, name: str, mediator: ChatMediator):
        self.name = name
        self._mediator = mediator
        mediator.register(self)

    def send(self, recipient: str, msg: str):
        self._mediator.send(self.name, recipient, msg)

    def receive(self, sender: str, msg: str):
        print(f"[{self.name}] Received from {sender}: {msg!r}")

chat = ChatMediator()
alice = ChatUser('Alice', chat)
bob   = ChatUser('Bob',   chat)
carol = ChatUser('Carol', chat)

alice.send('Bob',  'Hey Bob!')
bob.send('all', 'Hello everyone!')

# ════════════════════════════════════════════════
# SPECIFICATION PATTERN — combinable business rules
# ════════════════════════════════════════════════
class Specification(ABC):
    @abstractmethod
    def is_satisfied_by(self, candidate) -> bool: ...

    def __and__(self, other): return AndSpec(self, other)
    def __or__(self, other):  return OrSpec(self, other)
    def __invert__(self):      return NotSpec(self)

class AndSpec(Specification):
    def __init__(self, a, b): self.a, self.b = a, b
    def is_satisfied_by(self, c): return self.a.is_satisfied_by(c) and self.b.is_satisfied_by(c)

class OrSpec(Specification):
    def __init__(self, a, b): self.a, self.b = a, b
    def is_satisfied_by(self, c): return self.a.is_satisfied_by(c) or self.b.is_satisfied_by(c)

class NotSpec(Specification):
    def __init__(self, a): self.a = a
    def is_satisfied_by(self, c): return not self.a.is_satisfied_by(c)

class IsAdult(Specification):
    def is_satisfied_by(self, user): return user.get('age', 0) >= 18

class IsPremium(Specification):
    def is_satisfied_by(self, user): return user.get('tier') == 'premium'

class IsActive(Specification):
    def is_satisfied_by(self, user): return user.get('active', False)

# Compose specs with operators:
can_access_premium_content = IsAdult() & IsPremium() & IsActive()
can_view_free_content       = IsActive() & (IsAdult() | ~IsPremium())

users = [
    {'name': 'Alice',  'age': 25, 'tier': 'premium', 'active': True},
    {'name': 'Bob',    'age': 16, 'tier': 'premium', 'active': True},
    {'name': 'Carol',  'age': 30, 'tier': 'free',    'active': False},
]

for user in users:
    can = can_access_premium_content.is_satisfied_by(user)
    print(f"{user['name']:6s}: premium_access={can}")

---
# 🔢 SECTION 24 — Advanced Dynamic Programming

In [ ]:
from functools import lru_cache

# ---- Rod Cutting — maximize revenue from cutting a rod ----
def rod_cutting(prices: list) -> int:
    n = len(prices)
    dp = [0] * (n + 1)
    for length in range(1, n + 1):
        for cut in range(1, length + 1):
            dp[length] = max(dp[length], prices[cut-1] + dp[length-cut])
    return dp[n]

# Prices for rods of length 1..8
prices = [1, 5, 8, 9, 10, 17, 17, 20]
print(f"Rod cutting (length 8): {rod_cutting(prices)}")  # 22

# ---- Egg Drop Problem — minimum trials to find critical floor ----
@lru_cache(maxsize=None)
def egg_drop(eggs: int, floors: int) -> int:
    """
    Returns minimum number of trials to find critical floor
    with 'eggs' eggs and 'floors' floors.
    """
    if eggs == 1: return floors     # Linear scan
    if floors <= 1: return floors   # 0 or 1 floors

    min_trials = float('inf')
    lo, hi = 1, floors

    # Binary search over floors for optimal split
    while lo <= hi:
        mid = (lo + hi) // 2
        breaks    = egg_drop(eggs-1, mid-1)  # Egg breaks
        no_breaks = egg_drop(eggs,   floors-mid)  # Egg survives
        worst = 1 + max(breaks, no_breaks)
        min_trials = min(min_trials, worst)
        if breaks < no_breaks: lo = mid + 1
        else: hi = mid - 1

    return min_trials

print(f"Egg drop (2 eggs, 10 floors): {egg_drop(2, 10)}")   # 4
print(f"Egg drop (3 eggs, 100 floors): {egg_drop(3, 100)}") # 9

# ---- Palindrome Partitioning — min cuts to make all parts palindromes ----
def min_palindrome_cuts(s: str) -> int:
    n = len(s)
    # is_pal[i][j] = True if s[i:j+1] is a palindrome
    is_pal = [[False]*n for _ in range(n)]

    for i in range(n): is_pal[i][i] = True
    for length in range(2, n+1):
        for i in range(n - length + 1):
            j = i + length - 1
            if s[i] == s[j]:
                is_pal[i][j] = (length == 2) or is_pal[i+1][j-1]

    # dp[i] = min cuts for s[0:i+1]
    dp = list(range(n))   # Worst case: cut at every char
    for i in range(1, n):
        if is_pal[0][i]:
            dp[i] = 0
        else:
            for j in range(1, i+1):
                if is_pal[j][i]:
                    dp[i] = min(dp[i], dp[j-1] + 1)

    return dp[n-1]

print(f"Palindrome cuts('aab'):        {min_palindrome_cuts('aab')}")         # 1
print(f"Palindrome cuts('racecarannex'):{min_palindrome_cuts('racecarannex')}")  # 2

In [ ]:
# ---- Burst Balloons — interval DP ----
def max_coins_balloons(nums: list) -> int:
    """
    Burst all balloons to maximize coins.
    Bursting balloon i earns nums[i-1]*nums[i]*nums[i+1].
    """
    nums = [1] + nums + [1]  # Add virtual boundary balloons
    n = len(nums)
    dp = [[0]*n for _ in range(n)]

    for length in range(2, n):
        for left in range(0, n - length):
            right = left + length
            for k in range(left+1, right):
                dp[left][right] = max(
                    dp[left][right],
                    nums[left] * nums[k] * nums[right] + dp[left][k] + dp[k][right]
                )

    return dp[0][n-1]

print(f"Burst balloons [3,1,5,8]: {max_coins_balloons([3,1,5,8])}")  # 167

# ---- Stock trading with cooldown — state machine DP ----
def max_profit_cooldown(prices: list) -> int:
    """
    Max profit with cooldown: after selling, must wait 1 day.
    States: HELD (own stock), SOLD (just sold), REST (cooldown/ready)
    """
    held, sold, rest = -float('inf'), 0, 0

    for price in prices:
        prev_held, prev_sold, prev_rest = held, sold, rest
        held = max(prev_held, prev_rest - price)  # Buy from rest state
        sold = prev_held + price                  # Sell from held state
        rest = max(prev_rest, prev_sold)           # Rest or continue resting

    return max(sold, rest)

print(f"Stock cooldown [1,2,3,0,2]: {max_profit_cooldown([1,2,3,0,2])}")  # 3

# ---- Wildcard Matching — regex-like DP ----
def wildcard_match(s: str, p: str) -> bool:
    """'*' matches any sequence, '?' matches any single char."""
    m, n = len(s), len(p)
    dp = [[False]*(n+1) for _ in range(m+1)]
    dp[0][0] = True

    for j in range(1, n+1):
        dp[0][j] = dp[0][j-1] and p[j-1] == '*'

    for i in range(1, m+1):
        for j in range(1, n+1):
            if p[j-1] == '*':
                dp[i][j] = dp[i-1][j] or dp[i][j-1]  # Match one+ or zero chars
            elif p[j-1] == '?' or s[i-1] == p[j-1]:
                dp[i][j] = dp[i-1][j-1]

    return dp[m][n]

tests = [('aa','a',False),('aa','*',True),('cb','?a',False),('adceb','*a*b',True)]
for s, p, expected in tests:
    result = wildcard_match(s, p)
    status = '✓' if result == expected else '✗'
    print(f"{status} match({s!r}, {p!r}) = {result}")

---
# 🎭 SECTION 25 — Enum, NamedTuple & Advanced Built-ins

In [ ]:
from enum import Enum, IntEnum, Flag, IntFlag, auto, unique

# ---- Basic Enum ----
@unique  # Ensures no duplicate values
class Direction(Enum):
    NORTH = 'N'
    SOUTH = 'S'
    EAST  = 'E'
    WEST  = 'W'

    @property
    def opposite(self):
        opposites = {Direction.NORTH: Direction.SOUTH,
                     Direction.SOUTH: Direction.NORTH,
                     Direction.EAST:  Direction.WEST,
                     Direction.WEST:  Direction.EAST}
        return opposites[self]

    def __str__(self): return self.value

d = Direction.NORTH
print(d, d.name, d.value, d.opposite)
print(Direction('S'))          # Direction.SOUTH — lookup by value
print(Direction['EAST'])       # Direction.EAST  — lookup by name
print(list(Direction))         # All members

# ---- auto() — auto-numbered values ----
class Priority(IntEnum):
    LOW    = auto()  # 1
    MEDIUM = auto()  # 2
    HIGH   = auto()  # 3
    URGENT = auto()  # 4

print(Priority.HIGH > Priority.LOW)    # True — IntEnum supports comparison
print(sorted([Priority.URGENT, Priority.LOW, Priority.HIGH]))  # [LOW, HIGH, URGENT]

# ---- Flag — bitfield enums (permissions, features) ----
class Permission(Flag):
    READ    = auto()
    WRITE   = auto()
    EXECUTE = auto()
    ALL     = READ | WRITE | EXECUTE  # Combination

user_perms = Permission.READ | Permission.WRITE
print(user_perms)                               # Permission.READ|WRITE
print(Permission.READ in user_perms)             # True
print(Permission.EXECUTE in user_perms)          # False
print(user_perms & Permission.WRITE)            # Permission.WRITE

# Grant execute
user_perms |= Permission.EXECUTE
print(user_perms == Permission.ALL)             # True

In [ ]:
# ---- vars, dir, getattr, setattr, hasattr — introspection ----
class MyClass:
    class_var = 42
    def __init__(self): self.instance_var = 'hello'
    def method(self): pass

obj = MyClass()
print(vars(obj))          # {'instance_var': 'hello'}
print(vars(MyClass))      # Class __dict__
print([x for x in dir(obj) if not x.startswith('_')])

# Dynamic attribute access
attr_name = 'instance_var'
print(getattr(obj, attr_name))               # 'hello'
setattr(obj, attr_name, 'world')
print(hasattr(obj, 'method'))                # True
delattr(obj, 'instance_var')
print(hasattr(obj, 'instance_var'))          # False

# ---- zip, map, filter, enumerate, sorted with key ----
names  = ['Charlie', 'Alice', 'Bob']
scores = [85, 92, 78]

# zip creates pairs
leaderboard = sorted(zip(names, scores), key=lambda x: x[1], reverse=True)
for rank, (name, score) in enumerate(leaderboard, 1):
    print(f"{rank}. {name}: {score}")

# ---- operator module — faster than lambdas ----
import operator

data = [{'name': 'Alice', 'age': 30}, {'name': 'Bob', 'age': 25}, {'name': 'Carol', 'age': 35}]
by_age  = sorted(data, key=operator.itemgetter('age'))
by_name = sorted(data, key=operator.itemgetter('name'))

print([d['name'] for d in by_age])   # ['Bob', 'Alice', 'Carol']
print([d['name'] for d in by_name])  # ['Alice', 'Bob', 'Carol']

# ---- functools.reduce for complex aggregations ----
from functools import reduce

# Flatten a list of lists
nested = [[1,2],[3,4],[5,6]]
flat   = reduce(operator.add, nested)
print(flat)  # [1,2,3,4,5,6]

# Deep get from nested dict
def deep_get(d: dict, *keys, default=None):
    return reduce(lambda acc, k: acc.get(k, default) if isinstance(acc, dict) else default,
                  keys, d)

config = {'db': {'primary': {'host': 'localhost', 'port': 5432}}}
print(deep_get(config, 'db', 'primary', 'host'))   # localhost
print(deep_get(config, 'db', 'replica', 'host'))   # None

---
# 📦 SECTION 26 — The Import System & Module Internals

In [ ]:
import sys
import importlib
import types

# ---- How imports work ----
# 1. Check sys.modules (cache) — return if found
# 2. Find module (sys.meta_path finders)
# 3. Load module (execute its code)
# 4. Cache in sys.modules

print(f"sys.modules has {len(sys.modules)} cached modules")
print(f"json in cache: {'json' in sys.modules}")

# ---- sys.path — where Python looks for modules ----
print("\nsys.path entries:")
for p in sys.path[:5]:
    print(f"  {p}")

# ---- Dynamic import ----
module_name = 'json'
json_mod = importlib.import_module(module_name)
print(f"\nDynamically imported: {json_mod.__name__}")

# ---- Reload a module ----
import json
importlib.reload(json)  # Re-executes module code

# ---- __import__ — low-level ----
os_mod = __import__('os')
print(os_mod.getcwd())

# ---- Creating a module object programmatically ----
virtual_module = types.ModuleType('virtual_math')
virtual_module.__doc__ = 'A dynamically created math module'
virtual_module.PI = 3.14159
virtual_module.square = lambda x: x * x

# Register in sys.modules so it can be imported
sys.modules['virtual_math'] = virtual_module

import virtual_math
print(f"PI = {virtual_math.PI}, 5² = {virtual_math.square(5)}")

In [ ]:
# ---- Custom Import Hook (Meta Path Finder) ----
# Import hooks let you intercept and customize the import system.

import importlib.abc
import importlib.machinery

class DebugImportFinder(importlib.abc.MetaPathFinder):
    """Logs every module import attempt."""

    _log = []

    def find_spec(self, fullname, path, target=None):
        DebugImportFinder._log.append(fullname)
        return None  # Don't handle it; let normal finders proceed

# Install the hook
debug_finder = DebugImportFinder()
sys.meta_path.insert(0, debug_finder)

import csv  # This triggers our hook
import pathlib

# Remove the hook
sys.meta_path.remove(debug_finder)

print("Modules attempted during import:", DebugImportFinder._log[:10])

# ---- __all__ — control what 'from module import *' exports ----
# In a module file, define __all__ to restrict star imports

# Simulating __all__ effect
class FakeModule:
    __all__ = ['public_func', 'PublicClass']

    def public_func(self): pass
    def _private_func(self): pass
    class PublicClass: pass
    class _PrivateClass: pass

# ---- Lazy imports — defer until needed ----
class LazyModule:
    """Wraps a module, only importing it on first attribute access."""

    def __init__(self, module_name: str):
        object.__setattr__(self, '_module_name', module_name)
        object.__setattr__(self, '_module', None)

    def _load(self):
        if object.__getattribute__(self, '_module') is None:
            name = object.__getattribute__(self, '_module_name')
            print(f"Lazily importing {name}...")
            object.__setattr__(self, '_module', importlib.import_module(name))
        return object.__getattribute__(self, '_module')

    def __getattr__(self, name):
        return getattr(self._load(), name)

lazy_numpy = LazyModule('json')  # Doesn't import yet
print("LazyModule created")
result = lazy_numpy.dumps({'key': 'value'})  # Import happens NOW
print(result)

---
# ✅ SECTION 27 — Python Best Practices, Idioms & Anti-Patterns

In [ ]:
# ════════════════ PYTHONIC IDIOMS ════════════════

# ---- 1. Unpacking ----
first, *middle, last = range(10)
print(f"first={first}, middle len={len(middle)}, last={last}")

a, b = b, a = 1, 2   # Swap without temp var

# Nested unpacking
(x, y), z = (1, 2), 3
print(x, y, z)  # 1 2 3

# ---- 2. Walrus operator := (Python 3.8+) ----
data = [1, 5, 2, 8, 3, 7]
if (n := len(data)) > 5:
    print(f"Processing {n} items")  # n already computed

# Useful in while loops:
import io
buf = io.BytesIO(b'Hello World')
while chunk := buf.read(4):
    print(f"Read chunk: {chunk}")

# ---- 3. Dict merge operators (Python 3.9+) ----
defaults = {'timeout': 30, 'retries': 3, 'debug': False}
overrides = {'timeout': 60, 'debug': True}

merged = defaults | overrides  # New dict, overrides wins
print(merged)

defaults |= overrides  # In-place update

# ---- 4. f-string advanced formatting ----
import math
pi = math.pi
name = 'Alice'
balance = 1234.5678

print(f"{pi:.4f}")           # 3.1416
print(f"{pi!r}")             # repr(pi)
print(f"{1_000_000:,}")      # 1,000,000 (thousands separator)
print(f"{balance:.2f}")      # 1234.57
print(f"{42:08b}")           # 00101010 (binary, zero-padded)
print(f"{name:>20}")         # Right-aligned in 20 chars
print(f"{name:^20}")         # Centered
print(f"{name:*<20}")        # Left-aligned, filled with *

# Debug format (Python 3.8+)
x = 42
print(f"{x=}")               # x=42 — great for debugging!

In [ ]:
# ════════════════ ANTI-PATTERNS & FIXES ════════════════

# ---- 1. Type checking anti-patterns ----
def process(data):
    # WRONG: type() is not polymorphic
    if type(data) == list:
        pass

    # CORRECT: isinstance supports inheritance
    if isinstance(data, (list, tuple)):
        pass

    # BEST: duck typing — if it walks like a list
    try:
        for item in data:  # Works for list, tuple, generator, etc.
            pass
    except TypeError:
        pass

# ---- 2. Exception handling anti-patterns ----
# WRONG: bare except catches everything including SystemExit, KeyboardInterrupt
try:
    pass
except:  # Never do this!
    pass

# CORRECT: catch specific exceptions
try:
    pass
except (ValueError, TypeError) as e:
    print(f"Handled: {e}")

# WRONG: silently swallowing exceptions
try:
    result = int('abc')
except ValueError:
    pass  # Now result is undefined — bug waiting to happen!

# CORRECT: provide a meaningful default
try:
    result = int('abc')
except ValueError:
    result = 0  # Or raise, or log and re-raise

# ---- 3. N+1 query pattern (DB) ----
# WRONG: fetch users, then for EACH user fetch their orders (N+1 queries)
# users = User.query.all()
# for user in users:
#     orders = Order.query.filter_by(user_id=user.id).all()  # N extra queries!

# CORRECT: JOIN or eager load in one query
# users = User.query.options(joinedload(User.orders)).all()

# ---- 4. Repeated attribute lookup ----
import time

class HeavyObject:
    def __init__(self): self.data = list(range(1000))

obj = HeavyObject()

# WRONG: repeating obj.data.append in a tight loop (2 lookups each iter)
start = time.perf_counter()
for i in range(1000):
    obj.data.append(i)  # Looks up 'data' attr each iteration
t1 = time.perf_counter() - start

# CORRECT: cache the reference
obj2 = HeavyObject()
start = time.perf_counter()
append = obj2.data.append  # Cache the bound method
for i in range(1000):
    append(i)  # Direct call, no attribute lookup
t2 = time.perf_counter() - start

print(f"Without caching: {t1*1000:.3f}ms")
print(f"With    caching: {t2*1000:.3f}ms")
print(f"Speedup: {t1/t2:.1f}x")

In [ ]:
# ════════════════ PRODUCTION-GRADE PATTERNS ════════════════

# ---- Configuration management with dataclasses ----
import os
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class DatabaseConfig:
    host:     str   = field(default_factory=lambda: os.getenv('DB_HOST', 'localhost'))
    port:     int   = field(default_factory=lambda: int(os.getenv('DB_PORT', '5432')))
    name:     str   = field(default_factory=lambda: os.getenv('DB_NAME', 'mydb'))
    password: str   = field(default_factory=lambda: os.getenv('DB_PASS', ''))
    pool_size: int  = 10

    @property
    def url(self) -> str:
        return f"postgresql://{self.host}:{self.port}/{self.name}"

db_cfg = DatabaseConfig()
print(f"DB URL: {db_cfg.url}")

# ---- Rate limiter using token bucket ----
import time
import threading

class TokenBucket:
    """Thread-safe token bucket rate limiter."""

    def __init__(self, rate: float, capacity: float):
        self.rate      = rate      # tokens per second
        self.capacity  = capacity  # max tokens
        self.tokens    = capacity
        self.last_time = time.monotonic()
        self._lock     = threading.Lock()

    def consume(self, tokens: float = 1.0) -> bool:
        with self._lock:
            now = time.monotonic()
            elapsed = now - self.last_time
            self.last_time = now

            # Refill tokens
            self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)

            if self.tokens >= tokens:
                self.tokens -= tokens
                return True
            return False

    def __call__(self, func):
        """Use as a decorator."""
        import functools
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if not self.consume():
                raise RuntimeError("Rate limit exceeded")
            return func(*args, **kwargs)
        return wrapper

limiter = TokenBucket(rate=5, capacity=5)  # 5 requests/sec

@limiter
def api_call(n):
    return f"Response {n}"

for i in range(7):
    try:
        print(api_call(i))
    except RuntimeError as e:
        print(f"Request {i}: {e}")
    time.sleep(0.1)

In [ ]:
# ---- Dependency Injection Container ----
import inspect
from typing import get_type_hints

class DIContainer:
    """Simple dependency injection container."""

    def __init__(self):
        self._factories: dict = {}
        self._singletons: dict = {}

    def register(self, interface, factory, singleton=True):
        self._factories[interface] = (factory, singleton)

    def resolve(self, interface):
        if interface not in self._factories:
            raise KeyError(f"No factory registered for {interface}")

        factory, is_singleton = self._factories[interface]

        if is_singleton and interface in self._singletons:
            return self._singletons[interface]

        # Auto-resolve constructor dependencies
        hints = get_type_hints(factory.__init__) if hasattr(factory, '__init__') else {}
        params = inspect.signature(factory).parameters

        deps = {}
        for name, param in params.items():
            if name == 'self': continue
            if param.annotation in self._factories:
                deps[name] = self.resolve(param.annotation)

        instance = factory(**deps)

        if is_singleton:
            self._singletons[interface] = instance

        return instance

# Usage
class IEmailService: pass
class IUserRepo: pass

class EmailService(IEmailService):
    def send(self, to, msg): print(f"Email to {to}: {msg}")

class UserRepo(IUserRepo):
    def get(self, user_id): return {'id': user_id, 'email': f'user{user_id}@example.com'}

class UserService:
    def __init__(self, repo: IUserRepo, email: IEmailService):
        self.repo  = repo
        self.email = email

    def notify(self, user_id: int, message: str):
        user = self.repo.get(user_id)
        self.email.send(user['email'], message)

# Wire it up
container = DIContainer()
container.register(IEmailService, EmailService)
container.register(IUserRepo,     UserRepo)
container.register(UserService,   lambda repo=None, email=None: UserService(
    container.resolve(IUserRepo), container.resolve(IEmailService)
))

svc = container.resolve(UserService)
svc.notify(42, 'Welcome to the platform!')

---
## 🎓 Expert Python Mastery — Summary

| Section | Expert Concepts Covered |
|---------|------------------------|
| **Memory Management** | Reference counting, cyclic GC, weak references, `weakref.finalize`, `tracemalloc` |
| **CPython Internals** | `dis`, bytecode, code objects, `__getattr__` vs `__getattribute__`, data model |
| **Advanced Generators** | Pipelines, `send()`/`throw()`/`close()`, return values, `yield from` |
| **Advanced asyncio** | Cancellation, timeouts, semaphores, events, backpressure, streams |
| **Performance** | `cProfile`, `timeit`, `tracemalloc`, optimization techniques, `array`, `__slots__` |
| **Graph Algorithms** | BFS, DFS, Dijkstra, Bellman-Ford, Floyd-Warshall, Kahn's sort, Kruskal's MST, Union-Find |
| **Tree Structures** | BST, Trie, Segment Tree |
| **Sorting & Bits** | Quicksort, Mergesort, Counting Sort, binary search variants, bitmask DP (TSP) |
| **Testing** | pytest fixtures, parametrize, `Mock`, `patch`, `MagicMock` |
| **Regex** | Named groups, lookahead/behind, verbose mode, `sub` with callable, compiled patterns |
| **File I/O** | `pathlib`, JSON custom encoding, `pickle` custom state, CSV |
| **Logging** | JSON formatter, rotating handlers, `LoggerAdapter`, structured logging |
| **Networking** | TCP/UDP sockets, async TCP with `asyncio.start_server` |
| **Type System** | `Generic`, `TypeVar`, `overload`, `TypeGuard`, `Annotated`, `Literal`, `Final`, `ParamSpec` |
| **Pattern Matching** | Class, sequence, mapping, OR, wildcard, guard patterns |
| **Advanced OOP** | Mixins, class factories, runtime `Protocol`, `@runtime_checkable` |
| **More Patterns** | Visitor, Null Object, Mediator, Specification |
| **Advanced DP** | Rod cutting, Egg drop, Palindrome partitioning, Burst balloons, Wildcard matching |
| **Enum** | `Flag`, `IntEnum`, `auto()`, `@unique`, bitfield operations |
| **Import System** | `sys.modules`, `importlib`, meta path finders, lazy modules |
| **Best Practices** | Idioms, anti-patterns, config management, rate limiter, DI container |

### 📚 Essential References
- [Python Data Model](https://docs.python.org/3/reference/datamodel.html)
- [CPython Internals (Book)](https://realpython.com/products/cpython-internals-book/)
- [Fluent Python 2nd Ed](https://www.oreilly.com/library/view/fluent-python-2nd/9781492056348/)
- [High Performance Python](https://www.oreilly.com/library/view/high-performance-python/9781492055013/)
- [Python Concurrency with asyncio](https://www.manning.com/books/python-concurrency-with-asyncio)
- [Architecture Patterns with Python](https://www.oreilly.com/library/view/architecture-patterns-with/9781492052197/)